In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
%pip install -q "openai>=1.40" pyyaml "azure-search-documents>=11.6"

StatementMeta(, 3b97830a-e7ad-4d6c-b260-b9ee5f7f8d97, 7, Finished, Available, Finished, True)


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: /nfs4/pyenv-faf58ce6-8ad6-4474-b80f-52931b4c9e8b/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [2]:
import os, re, glob, hashlib, time, yaml
from datetime import datetime, timezone
from openai import AzureOpenAI

# ---------- Settings ----------
AOAI_ENDPOINT    = "https://aoai-claimsiq.services.ai.azure.com/"   # <-- CHANGE (same as AZURE_OPENAI_ENDPOINT in .env)
AOAI_KEY         = "<paste-key-here>"                           # <-- CHANGE (same as AZURE_OPENAI_API_KEY). Remove before exporting!
AOAI_API_VERSION = "2024-10-21"
EMBED_DEPLOYMENT = "text-embedding-3-small"   # your real deployment name
EMBED_DIMS       = 1536

BRONZE_LOCAL = "/lakehouse/default/Files/bronze"   # path for normal Python (open, glob)
BRONZE_SPARK = "Files/bronze"                      # path for Spark (spark.read)
RUN_TS = datetime.now(timezone.utc).replace(tzinfo=None)

def save(df, name):
    """Write a Spark DataFrame as a Delta table, replacing any old version."""
    df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(name)
    print(f"saved {name}: {spark.table(name).count()} rows")

# ---------- Connection test ----------
client = AzureOpenAI(azure_endpoint=AOAI_ENDPOINT, api_key=AOAI_KEY, api_version=AOAI_API_VERSION)
r = client.embeddings.create(model=EMBED_DEPLOYMENT, input="Joint replacement waiting period")
print("Azure OpenAI connection: OK")
print("embedding dimensions:", len(r.data[0].embedding), "(expected 1536)")
print("tokens used:", r.usage.total_tokens)

StatementMeta(, 3b97830a-e7ad-4d6c-b260-b9ee5f7f8d97, 9, Finished, Available, Finished, False)

Azure OpenAI connection: OK
embedding dimensions: 1536 (expected 1536)
tokens used: 4


In [ ]:
# ---------- Cell 3: BRONZE layer - one row per raw document ----------
from pyspark.sql import Row

paths = sorted(glob.glob(f"{BRONZE_LOCAL}/docs/**/*.md", recursive=True))
print("markdown files found:", len(paths))          # expect 50

raw_rows = []
for p in paths:
    text = open(p, encoding="utf-8").read()
    raw_rows.append(Row(
        file_path    = p.replace("/lakehouse/default/", ""),   # e.g. Files/bronze/docs/wordings/HG-WORD-2025.md
        folder       = p.split("/")[-2],                       # e.g. wordings
        file_name    = os.path.basename(p),                    # e.g. HG-WORD-2025.md
        raw_text     = text,                                   # the file exactly as uploaded
        content_hash = hashlib.sha256(text.encode("utf-8")).hexdigest(),
        ingested_at  = RUN_TS,
    ))

save(spark.createDataFrame(raw_rows), "bronze_raw_documents")

StatementMeta(, fed4a50c-51fa-4bac-9b7d-00a0e767b94f, 23, Finished, Available, Finished, False)

markdown files found: 50


saved bronze_raw_documents: 50 rows


In [ ]:
# ---------- Check the bronze table ----------
b = spark.table("bronze_raw_documents")

print("Rows per folder:")
b.groupBy("folder").count().orderBy("folder").show()

print("Every file starts with '---':",
      b.filter(~b.raw_text.rlike(r"^\ufeff?---")).count() == 0)
print("Unique hashes:", b.select("content_hash").distinct().count(), "(expect 50)")

display(b.select("folder", "file_name", "content_hash", "ingested_at").limit(5))

StatementMeta(, fed4a50c-51fa-4bac-9b7d-00a0e767b94f, 24, Finished, Available, Finished, False)

Rows per folder:


+------------+-----+
|      folder|count|
+------------+-----+
|   circulars|    3|
|         faq|    9|
|       fraud|    3|
|       memos|    5|
|      riders|    4|
|     scripts|    8|
|        sops|   10|
|underwriting|    5|
|    wordings|    3|
+------------+-----+



Every file starts with '---': True
Unique hashes: 50 (expect 50)


SynapseWidget(Synapse.DataFrame, 208d58ed-b2d4-4b29-82e3-1126ac89ba52)

In [ ]:
# ---------- Cell 5: parsing functions (v2: tolerates unquoted colons in YAML values) ----------
FRONT_MATTER = re.compile(r"^\ufeff?---\s*\n(.*?)\n---\s*\n(.*)$", re.S)
HEADING = re.compile(r"^## (Clause|Section) (\d+(?:\.\d+)*)\.?\s+(.+)$", re.M)
YAML_LINE = re.compile(r"^(\w+):\s+(.+)$")

def quote_unsafe_values(header):
    """Put quotes around values like 'NIR Circular Summary: Claim ...' that contain ': '."""
    fixed = []
    for line in header.splitlines():
        m = YAML_LINE.match(line)
        if m:
            key, value = m.group(1), m.group(2).strip()
            if ": " in value and not value.startswith(('"', "'", "[")):
                line = f'{key}: "{value.replace(chr(34), chr(92) + chr(34))}"'
        fixed.append(line)
    return "\n".join(fixed)

def parse_doc(text):
    """Split a file into (metadata dict, body text)."""
    m = FRONT_MATTER.match(text)
    if not m:
        raise ValueError("missing or broken YAML header")
    return yaml.safe_load(quote_unsafe_values(m.group(1))), m.group(2)

def split_sections(body):
    """Yield (heading_type, number, heading_title, full_text) for every ## heading."""
    found = list(HEADING.finditer(body))
    for i, m in enumerate(found):
        end = found[i + 1].start() if i + 1 < len(found) else len(body)
        yield m.group(1), m.group(2), m.group(3).strip(), body[m.start():end].strip()

def iso_date(v):
    s = str(v)[:10]
    datetime.strptime(s, "%Y-%m-%d")
    return s

# ---------- Self-test on one of the files that failed ----------
row = spark.table("bronze_raw_documents").filter("file_name = 'CIR-01.md'").first()
meta, body = parse_doc(row.raw_text)
print("title:", meta["title"])
print("sections:", [s[1] for s in split_sections(body)])

StatementMeta(, fed4a50c-51fa-4bac-9b7d-00a0e767b94f, 25, Finished, Available, Finished, False)

title: NIR Circular Summary: Claim Settlement Timelines
sections: ['1', '2', '3', '4']


In [ ]:
# ---------- Cell 6: SILVER layer - documents + clause chunks ----------
import re, hashlib
from pyspark.sql import Row
from pyspark.sql import functions as F

doc_rows, chunk_rows, problems = [], [], []

for r in spark.table("bronze_raw_documents").collect():
    try:
        meta, body = parse_doc(r.raw_text)
    except Exception as e:
        problems.append(f"{r.file_name}: {e}")
        continue

    doc_id   = meta["doc_id"]
    audience = [a.strip().lower() for a in meta["audience"]]
    eff_from = iso_date(meta["effective_from"])
    eff_to   = iso_date(meta["effective_to"])
    sections = list(split_sections(body))

    if not body.lstrip().startswith("## "):
        problems.append(f"{doc_id}: text before first heading (would be lost)")
    if not sections:
        problems.append(f"{doc_id}: no headings found")

    # one row per document
    doc_rows.append(Row(
        doc_id=doc_id, title=meta["title"], product=meta["product"], version=str(meta["version"]),
        doc_type=meta["doc_type"], audience=audience, sensitivity=meta["sensitivity"],
        effective_from=eff_from, effective_to=eff_to, section_count=len(sections),
        source_path=r.file_path, content_hash=r.content_hash))

    # one row per clause/section, every row carries the document's labels
    for kind, num, heading_title, content in sections:
        chunk_rows.append(Row(
            chunk_id       = f"{doc_id}__{num.replace('.', '_')}",   # e.g. HG-WORD-2025__4_2 (safe as a Search key)
            doc_id         = doc_id,
            title          = meta["title"],
            citation       = f"{doc_id}§{num}",                      # e.g. HG-WORD-2025§4.2 (same as benchmark.csv)
            heading_type   = kind,
            clause_number  = num,
            heading_title  = heading_title,
            content        = content,
            embed_text     = f"{meta['title']} | product: {meta['product']} | version: {meta['version']}\n{content}",   # FIXED: space added
            word_count     = len(content.split()),
            product        = meta["product"],
            version        = str(meta["version"]),
            doc_type       = meta["doc_type"],
            audience       = audience,
            sensitivity    = meta["sensitivity"],
            effective_from = eff_from,
            effective_to   = eff_to,
            source_path    = r.file_path,
            chunk_hash     = hashlib.sha256(content.encode("utf-8")).hexdigest()))

to_dates = lambda df: (df.withColumn("effective_from", F.to_date("effective_from"))
                         .withColumn("effective_to",   F.to_date("effective_to")))

save(to_dates(spark.createDataFrame(doc_rows)),   "silver_documents")
save(to_dates(spark.createDataFrame(chunk_rows)), "silver_chunks")
print("problems:", problems or "none")

# quick proof the fix worked
print(spark.table("silver_chunks").filter("citation = 'HG-WORD-2025§4.2'").first().embed_text.splitlines()[0])

StatementMeta(, fed4a50c-51fa-4bac-9b7d-00a0e767b94f, 32, Finished, Available, Finished, False)

saved silver_documents: 50 rows


saved silver_chunks: 254 rows
problems: none
Health Gold Policy Wording 2025 | product: HEALTH_GOLD | version: v2025


In [ ]:
# ---------- Cell 7: FULL QUALITY CHECKS on silver ----------
import re
from pyspark.sql import functions as F

docs   = spark.table("silver_documents")
chunks = spark.table("silver_chunks")
rows   = chunks.collect()
c      = {r.citation: r.content for r in rows}        # citation -> clause text
errs   = []
def must(cond, msg):
    if not cond: errs.append(msg)

def subline(citation, sub):
    """Return the line that starts with a sub-clause number, e.g. '4.2.1 ...'."""
    m = re.search(rf"^{re.escape(sub)}\b.*$", c.get(citation, ""), re.M)
    return m.group(0) if m else ""

# ---- A. Counts ----
must(docs.count() == 50, f"A: expected 50 documents, got {docs.count()}")
must(len(rows) == 254,   f"A: expected 254 chunks, got {len(rows)}")
expected_types = {"wording": 78, "sop": 41, "faq": 36, "script": 24, "uw_guideline": 20,
                  "rider": 16, "memo": 15, "circular": 12, "fraud_playbook": 12}
actual_types = {r["doc_type"]: r["count"] for r in chunks.groupBy("doc_type").count().collect()}
must(actual_types == expected_types, f"A: chunks per doc_type differ: {actual_types}")

# ---- B. Unique IDs, no tiny chunks ----
ids = [r.chunk_id for r in rows]
must(len(ids) == len(set(ids)), "B: duplicate chunk_ids found")
tiny = [r.citation for r in rows if r.word_count < 20]
must(not tiny, f"B: chunks under 20 words: {tiny}")

# ---- C. Allowed label values ----
PRODUCTS = {"HEALTH_GOLD", "HEALTH_SILVER"}
SENS     = {"General", "Internal", "Restricted"}
TYPES    = set(expected_types) | {"approved_guidance"}
AUD      = {"adjuster", "service", "underwriter", "fraud"}
for r in rows:
    must(r.product in PRODUCTS,       f"C: {r.citation} bad product {r.product!r}")
    must(r.sensitivity in SENS,       f"C: {r.citation} bad sensitivity {r.sensitivity!r}")
    must(r.doc_type in TYPES,         f"C: {r.citation} bad doc_type {r.doc_type!r}")
    must(set(r.audience) <= AUD,      f"C: {r.citation} bad audience {r.audience}")
    must(r.effective_from <= r.effective_to, f"C: {r.citation} effective_from after effective_to")
    must(f"product: {r.product} | version: {r.version}" in r.embed_text, f"C: {r.citation} embed_text label missing")
# ---- D. Manifest rules ----
for r in rows:
    if r.doc_type == "fraud_playbook":
        must(r.sensitivity == "Restricted" and "service" not in r.audience,
             f"D: {r.citation} fraud chunk must be Restricted and hidden from service")
    if r.doc_type == "uw_guideline":
        must(list(r.audience) == ["underwriter"], f"D: {r.citation} UW chunk audience should be [underwriter]")
silver_docs = {r.doc_id for r in rows if r.product == "HEALTH_SILVER"}
must(silver_docs == {"HS-WORD-2025", "FAQ-SILVER-01", "FAQ-SILVER-02", "FAQ-SILVER-03"},
     f"D: HEALTH_SILVER documents are {sorted(silver_docs)}")
d = {r.doc_id: r for r in docs.collect()}
must(str(d["HG-WORD-2024"].effective_to) == "2024-12-31", "D: HG-WORD-2024 should end 2024-12-31")
must(str(d["HG-WORD-2025"].effective_from) == "2025-01-01", "D: HG-WORD-2025 should start 2025-01-01")
must(str(d["MEMO-01"].effective_to) == "2023-12-31", "D: superseded MEMO-01 should end 2023-12-31")

# ---- E. Banned words and fraud isolation ----
BANNED      = ["lithium", "li-ion", "battery", "batteries", "overseas", "international", "abroad", "evacuation"]
FRAUD_TERMS = ["90 days", "90-day", "150%", "red-flag score", "watchlist"]
for r in rows:
    low = r.content.lower()
    if r.doc_type != "approved_guidance":
        for w in BANNED:
            if w in low: errs.append(f"E: {r.citation} contains banned word '{w}'")
    if r.doc_type != "fraud_playbook":
        for t in FRAUD_TERMS:
            if t in low: errs.append(f"E: {r.citation} contains fraud term '{t}'")

# ---- F. Planted facts (the demos depend on these) ----
must("24 months" in subline("HG-WORD-2024§4.2", "4.2.1"), "F: v2024 knee (4.2.1) should be 24 months")
must("12 months" in subline("HG-WORD-2025§4.2", "4.2.1"), "F: v2025 knee (4.2.1) should be 12 months")
must("36 months" in subline("HS-WORD-2025§4.2", "4.2.1"), "F: Silver knee (4.2.1) should be 36 months")
must("48 months" in subline("HG-WORD-2024§4.3", "4.3.1"), "F: v2024 PED (4.3.1) should be 48 months")
must("36 months" in subline("HG-WORD-2025§4.3", "4.3.1"), "F: v2025 PED (4.3.1) should be 36 months")
must("1%" in c.get("HG-WORD-2024§8.1", "") and "2%" in c.get("HG-WORD-2025§8.1", ""), "F: room rent should be 1% vs 2%")
# FIXED: check the words, not the sub-clause number (v2025 renumbered 5.4.2 -> 5.4.3)
d24 = c.get("HG-WORD-2024§5.4", "").lower()
d25 = c.get("HG-WORD-2025§5.4", "").lower()
must("veneers" not in d24 and "teeth whitening" not in d24,
     "F: v2024 5.4 should NOT mention cosmetic dental (veneers / whitening)")
must("cosmetic dental" in d25 and "veneers" in d25,
     "F: v2025 5.4 should exclude cosmetic dental (veneers)")
must("50,000" in c.get("HG-WORD-2025§2.6", ""), "F: v2025 maternity should mention ₹50,000")
must("0 months" in c.get("RID-WPW-01§2", ""), "F: waiver rider §2 should reduce to 0 months")
must("5,00,000" in c.get("FRD-01§2", "") and "90 days" in c.get("FRD-01§2", ""), "F: FRD-01 §2 should hold the SIU threshold")
must("chief underwriter" in c.get("UW-OCC-01§4", "").lower(), "F: UW-OCC-01 §4 should say refer to Chief Underwriter")

# ---- Result ----
print(f"{len(rows)} chunks checked")
print("ALL CHECKS PASSED ✅" if not errs else f"{len(errs)} PROBLEMS:\n - " + "\n - ".join(errs))
chunks.select(F.min("word_count").alias("min_words"), F.round(F.avg("word_count")).alias("avg_words"),
              F.max("word_count").alias("max_words")).show()

StatementMeta(, fed4a50c-51fa-4bac-9b7d-00a0e767b94f, 34, Finished, Available, Finished, False)

254 chunks checked
ALL CHECKS PASSED ✅
+---------+---------+---------+
|min_words|avg_words|max_words|
+---------+---------+---------+
|       87|    214.0|      922|
+---------+---------+---------+



In [ ]:
# ---------- Cell 8: GOLD reference tables (exact facts for the tools) ----------
from pyspark.sql import functions as F

def read_csv(name):
    """Read one reference CSV, clean hidden characters from headers, and trim every value."""
    df = (spark.read.option("header", True).option("encoding", "UTF-8")
          .csv(f"{BRONZE_SPARK}/reference/{name}.csv"))
    for col in df.columns:                                   # remove hidden BOM / spaces in headers
        df = df.withColumnRenamed(col, col.replace("\ufeff", "").strip())
    for col in df.columns:                                   # trim spaces around every value
        df = df.withColumn(col, F.trim(F.col(col)))
    return df

def nulls_if_blank(df, col):
    return df.withColumn(col, F.when(F.col(col) == "", None).otherwise(F.col(col)))

gold = {
  "gold_rejection_codes": read_csv("rejection_codes"),

  "gold_waiting_periods": nulls_if_blank(read_csv("waiting_periods"), "waiting_months")
        .withColumn("covered",        F.lower("covered") == "true")
        .withColumn("waiting_months", F.col("waiting_months").cast("int")),

  "gold_products": read_csv("products")
        .withColumn("effective_from",      F.to_date("effective_from"))
        .withColumn("effective_to",        F.to_date("effective_to"))
        .withColumn("room_rent_pct_of_si", F.col("room_rent_pct_of_si").cast("int")),

  "gold_sub_limits": read_csv("sub_limits")
        .withColumn("limit_inr", F.col("limit_inr").cast("int")),

  "gold_personas": read_csv("personas")
        .withColumn("allowed_audience", F.split("allowed_audience", r"\|")),

  "gold_sample_claims": read_csv("sample_claims")
        .withColumn("policy_start",      F.to_date("policy_start"))
        .withColumn("date_of_loss",      F.to_date("date_of_loss"))
        .withColumn("claimed_amount",    F.col("claimed_amount").cast("int"))
        .withColumn("sum_insured",       F.col("sum_insured").cast("int"))
        .withColumn("room_rent_per_day", F.col("room_rent_per_day").cast("int")),
}

expected_rows = {"gold_rejection_codes": 25, "gold_waiting_periods": 21, "gold_products": 3,
                 "gold_sub_limits": 10, "gold_personas": 3, "gold_sample_claims": 10}

problems = []
for name, df in gold.items():
    save(df, name)
    n = spark.table(name).count()
    if n != expected_rows[name]:
        problems.append(f"{name}: {n} rows, expected {expected_rows[name]}")

# ---- Spot checks on the facts the demos depend on ----
wp = {(r["product"], r["version"], r["condition"]): r["waiting_months"]
      for r in spark.table("gold_waiting_periods").collect()}
if wp.get(("HEALTH_GOLD", "v2024", "joint_replacement")) != 24:   problems.append("knee v2024 should be 24")
if wp.get(("HEALTH_GOLD", "v2025", "joint_replacement")) != 12:   problems.append("knee v2025 should be 12")
if wp.get(("HEALTH_SILVER", "v2025", "joint_replacement")) != 36: problems.append("Silver knee should be 36")
if wp.get(("HEALTH_GOLD", "v2025", "pre_existing_disease")) != 36: problems.append("PED v2025 should be 36")

cl217 = spark.table("gold_rejection_codes").filter("code = 'CL-217'").collect()
if not cl217 or cl217[0]["customer_script_id"] != "SCR-PED-01":  problems.append("CL-217 row missing or wrong script")

svc = spark.table("gold_personas").filter("persona = 'service'").first()
if svc is None or "fraud" in svc["allowed_audience"]:            problems.append("service persona must NOT include fraud")

if spark.table("gold_sample_claims").filter("policy_start IS NULL OR date_of_loss IS NULL").count():
    problems.append("some sample_claims dates did not convert (not YYYY-MM-DD?)")

print("\nGOLD REFERENCE TABLES:", "ALL OK ✅" if not problems else "PROBLEMS:\n - " + "\n - ".join(problems))

print("\nKnee waiting periods (the version demo):")
spark.table("gold_waiting_periods").filter("condition = 'joint_replacement'") \
     .select("product", "version", "waiting_months", "clause_ref").show()

StatementMeta(, fed4a50c-51fa-4bac-9b7d-00a0e767b94f, 35, Finished, Available, Finished, False)

saved gold_rejection_codes: 25 rows


saved gold_waiting_periods: 21 rows


saved gold_products: 3 rows


saved gold_sub_limits: 10 rows


saved gold_personas: 3 rows


saved gold_sample_claims: 10 rows



GOLD REFERENCE TABLES: ALL OK ✅

Knee waiting periods (the version demo):
+-------------+-------+--------------+----------+
|      product|version|waiting_months|clause_ref|
+-------------+-------+--------------+----------+
|  HEALTH_GOLD|  v2024|            24|       4.2|
|  HEALTH_GOLD|  v2025|            12|       4.2|
|HEALTH_SILVER|  v2025|            36|       4.2|
+-------------+-------+--------------+----------+



In [ ]:
# ---------- Cell 9: GOLD chunk index - embeddings for every clause ----------
import time
from openai import AzureOpenAI
from pyspark.sql.types import (StructType, StructField, StringType, ArrayType,
                               FloatType, IntegerType, TimestampType)

client = AzureOpenAI(azure_endpoint=AOAI_ENDPOINT, api_key=AOAI_KEY, api_version=AOAI_API_VERSION)

def embed_batch(texts, retries=6):
    """Embed a list of texts; wait and retry on rate limits (429) or temporary errors."""
    for attempt in range(retries):
        try:
            r = client.embeddings.create(model=EMBED_DEPLOYMENT, input=texts)
            return [d.embedding for d in r.data], r.usage.total_tokens
        except Exception as e:
            wait = 5 * 2 ** attempt
            print(f"  retry in {wait}s: {str(e)[:120]}")
            time.sleep(wait)
    raise RuntimeError("embedding failed after retries")

rows = spark.table("silver_chunks").orderBy("chunk_id").collect()
print("chunks to embed:", len(rows))                       # expect 254

vectors, total_tokens, BATCH = [], 0, 16
start = time.time()
for i in range(0, len(rows), BATCH):
    vecs, toks = embed_batch([r.embed_text for r in rows[i:i + BATCH]])
    vectors += vecs
    total_tokens += toks
    print(f"embedded {min(i + BATCH, len(rows))}/{len(rows)}")

assert len(vectors) == len(rows), "number of vectors does not match number of chunks"
assert all(len(v) == EMBED_DIMS for v in vectors), "wrong vector size - check the embedding deployment"

cost = total_tokens / 1_000_000 * 0.02          # text-embedding-3-small: about $0.02 per 1M tokens
print(f"\ntokens used: {total_tokens:,} | approx cost: ${cost:.4f} | time: {time.time() - start:.0f}s")

# ---- Table shaped exactly like the Azure AI Search index (Phase 3) ----
schema = StructType([
    StructField("id",              StringType()),            # Search key, e.g. HG-WORD-2025__4_2
    StructField("doc_id",          StringType()),
    StructField("title",           StringType()),
    StructField("citation",        StringType()),            # e.g. HG-WORD-2025§4.2
    StructField("clause_number",   StringType()),
    StructField("heading_title",   StringType()),
    StructField("content",         StringType()),
    StructField("product",         StringType()),
    StructField("version",         StringType()),
    StructField("doc_type",        StringType()),
    StructField("audience",        ArrayType(StringType())),
    StructField("sensitivity",     StringType()),
    StructField("effective_from",  StringType()),            # '2025-01-01T00:00:00Z' (Search date format)
    StructField("effective_to",    StringType()),
    StructField("word_count",      IntegerType()),
    StructField("content_vector",  ArrayType(FloatType())),  # the 1,536 numbers
    StructField("embedding_model", StringType()),
    StructField("chunk_hash",      StringType()),
    StructField("embedded_at",     TimestampType()),
])

index_rows = [(
    r.chunk_id, r.doc_id, r.title, r.citation, r.clause_number, r.heading_title, r.content,
    r.product, r.version, r.doc_type, list(r.audience), r.sensitivity,
    f"{r.effective_from.isoformat()}T00:00:00Z",
    f"{r.effective_to.isoformat()}T00:00:00Z",
    int(r.word_count), [float(x) for x in v], EMBED_DEPLOYMENT, r.chunk_hash, RUN_TS)
    for r, v in zip(rows, vectors)]

save(spark.createDataFrame(index_rows, schema), "gold_chunk_index")

# ---- Quick checks ----
g = spark.table("gold_chunk_index")
print("rows:", g.count(), "(expected 254)")
print("unique ids:", g.select("id").distinct().count(), "(expected 254)")
print("vector size of first row:", len(g.first().content_vector), "(expected 1536)")
display(g.select("id", "citation", "version", "effective_from", "effective_to", "audience").limit(5))

StatementMeta(, fed4a50c-51fa-4bac-9b7d-00a0e767b94f, 36, Finished, Available, Finished, False)

chunks to embed: 254


embedded 16/254
embedded 32/254
embedded 48/254
embedded 64/254
embedded 80/254


embedded 96/254
embedded 112/254


embedded 128/254
embedded 144/254
embedded 160/254
embedded 176/254
embedded 192/254
embedded 208/254


embedded 224/254
embedded 240/254
embedded 254/254

tokens used: 76,005 | approx cost: $0.0015 | time: 7s


saved gold_chunk_index: 254 rows
rows: 254 (expected 254)
unique ids: 254 (expected 254)


vector size of first row: 1536 (expected 1536)


SynapseWidget(Synapse.DataFrame, fd601364-5dca-47ed-b519-1fc75c312bea)

In [ ]:
# ---------- Cell 10: confirm gold tables + test version and role filters ----------
from pyspark.sql import functions as F

results = []
def check(name, cond):
    results.append(cond)
    print(("PASS  " if cond else "FAIL  ") + name)

# ---- 0. Confirm Steps 8 and 9 ----
g = spark.table("gold_chunk_index")
check("gold_chunk_index has 254 rows", g.count() == 254)
check("every vector has 1536 numbers", g.filter(F.size("content_vector") != 1536).count() == 0)
for t, n in {"gold_rejection_codes": 25, "gold_waiting_periods": 21, "gold_products": 3,
             "gold_sub_limits": 10, "gold_personas": 3, "gold_sample_claims": 10}.items():
    check(f"{t} has {n} rows", spark.table(t).count() == n)

# ---- Filter helper: what a persona can see for a given date of loss ----
def visible(date_of_loss, persona, product="HEALTH_GOLD", extra="1=1"):
    aud = spark.table("gold_personas").filter(F.col("persona") == persona).first()["allowed_audience"]
    aud_sql = ", ".join(f"'{a}'" for a in aud)
    return spark.sql(f"""
        SELECT citation, doc_id, version, doc_type FROM silver_chunks
        WHERE product = '{product}'
          AND effective_from <= DATE'{date_of_loss}' AND effective_to >= DATE'{date_of_loss}'
          AND arrays_overlap(audience, array({aud_sql}))
          AND ({extra})""")

# ---- 1. Version-correct (knee demo) ----
k24 = [r.citation for r in visible("2024-09-15", "adjuster", extra="doc_type='wording' AND clause_number='4.2'").collect()]
k25 = [r.citation for r in visible("2025-09-15", "adjuster", extra="doc_type='wording' AND clause_number='4.2'").collect()]
check(f"2024 loss sees only the v2024 knee clause -> {k24}", k24 == ["HG-WORD-2024§4.2"])
check(f"2025 loss sees only the v2025 knee clause -> {k25}", k25 == ["HG-WORD-2025§4.2"])

# ---- 2. Role-aware (fraud block demo) ----
check("service sees 0 fraud chunks",
      visible("2025-06-01", "service", extra="doc_type='fraud_playbook'").count() == 0)
check("adjuster sees 12 fraud chunks",
      visible("2025-06-01", "adjuster", extra="doc_type='fraud_playbook'").count() == 12)
check("underwriter sees UW-OCC-01 (Li-ion demo document)",
      visible("2025-06-01", "underwriter", extra="doc_id='UW-OCC-01'").count() == 4)
check("service does NOT see UW-OCC-01",
      visible("2025-06-01", "service", extra="doc_id='UW-OCC-01'").count() == 0)

# ---- 3. Outdated and wrong-product content hidden ----
check("superseded MEMO-01 hidden for a 2024 loss",
      visible("2024-09-15", "adjuster", extra="doc_id='MEMO-01'").count() == 0)
check("Silver documents never appear in Gold queries",
      visible("2025-09-15", "adjuster", extra="doc_id LIKE 'HS-%' OR doc_id LIKE 'FAQ-SILVER%'").count() == 0)

print(f"\n{sum(results)}/{len(results)} passed",
      "- PHASE 2 DATA READY ✅" if all(results) else "- fix the FAIL lines before Phase 3")

StatementMeta(, fed4a50c-51fa-4bac-9b7d-00a0e767b94f, 38, Finished, Available, Finished, False)

PASS  gold_chunk_index has 254 rows
PASS  every vector has 1536 numbers


PASS  gold_rejection_codes has 25 rows
PASS  gold_waiting_periods has 21 rows
PASS  gold_products has 3 rows


PASS  gold_sub_limits has 10 rows
PASS  gold_personas has 3 rows
PASS  gold_sample_claims has 10 rows


PASS  2024 loss sees only the v2024 knee clause -> ['HG-WORD-2024§4.2']
PASS  2025 loss sees only the v2025 knee clause -> ['HG-WORD-2025§4.2']


PASS  service sees 0 fraud chunks
PASS  adjuster sees 12 fraud chunks
PASS  underwriter sees UW-OCC-01 (Li-ion demo document)


PASS  service does NOT see UW-OCC-01
PASS  superseded MEMO-01 hidden for a 2024 loss
PASS  Silver documents never appear in Gold queries

16/16 passed - PHASE 2 DATA READY ✅


In [3]:
# ---------- Cell 11: upload gold_chunk_index to Azure AI Search ----------
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient

SEARCH_ENDPOINT = "https://srch-claimsiq-vh.search.windows.net"
SEARCH_KEY      = "<paste-search-admin-key-here>"    # <-- CHANGE. Remove before exporting!
SEARCH_INDEX    = "claimsiq-clauses"

search = SearchClient(SEARCH_ENDPOINT, SEARCH_INDEX, AzureKeyCredential(SEARCH_KEY))

# Fields that exist in the index (embedded_at is in the table but NOT in the index)
INDEX_FIELDS = ["id", "doc_id", "title", "citation", "clause_number", "heading_title", "content",
                "product", "version", "doc_type", "sensitivity", "audience",
                "effective_from", "effective_to", "word_count",
                "content_vector", "embedding_model", "chunk_hash"]

rows = spark.table("gold_chunk_index").select(*INDEX_FIELDS).collect()
docs = []
for r in rows:
    d = r.asDict()
    d["audience"]       = list(d["audience"])
    d["content_vector"] = [float(x) for x in d["content_vector"]]
    docs.append(d)
print("documents to upload:", len(docs))          # expect 254

ok, failed = 0, []
for i in range(0, len(docs), 50):                 # batches of 50
    results = search.upload_documents(documents=docs[i:i + 50])
    for res in results:
        if res.succeeded: ok += 1
        else: failed.append((res.key, res.error_message))
    print(f"uploaded {min(i + 50, len(docs))}/{len(docs)}")

print(f"\nsucceeded: {ok} | failed: {len(failed)}")
for key, msg in failed[:5]: print("  FAILED", key, msg)

import time; time.sleep(5)                        # the document count takes a few seconds to update
print("documents in the index now:", search.get_document_count(), "(expected 254)")

StatementMeta(, 3b97830a-e7ad-4d6c-b260-b9ee5f7f8d97, 10, Finished, Available, Finished, False)

documents to upload: 254
uploaded 50/254
uploaded 100/254
uploaded 150/254
uploaded 200/254
uploaded 250/254
uploaded 254/254

succeeded: 254 | failed: 0
documents in the index now: 254 (expected 254)
